In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/carrie1/ecommerce-data/data.csv


In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv("/kaggle/input/datasets/carrie1/ecommerce-data/data.csv",encoding="ISO-8859-1")
df.info()
df.describe()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
#missing value check
print(df.isnull().sum())


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [5]:
df["Description"]=df["Description"].fillna("UNKNOWN")
df["CustomerID"]=df["CustomerID"].fillna("UNKNOWN")




In [6]:

print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [7]:
#Duplicate value check
print(df.duplicated().sum())

5268


In [8]:
#Remove duplicate value
df=df.drop_duplicates()
print("Shape after removing duplicates:",df.shape)

Shape after removing duplicates: (536641, 8)


In [9]:
# Convert columns to correct data types

df['InvoiceNo'] = df['InvoiceNo'].astype('string')

df['StockCode'] = df['StockCode'].astype('string')

df['Description'] = df['Description'].astype('string')

df['Quantity'] = pd.to_numeric(
    df['Quantity'], errors='coerce'
).astype('Int64')

df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    errors='coerce',
    dayfirst=True
)

df['UnitPrice'] = pd.to_numeric(
    df['UnitPrice'], errors='coerce'
)

df['CustomerID'] = pd.to_numeric(
    df['CustomerID'], errors='coerce'
).astype('Int64')

df['Country'] = df['Country'].astype('string')

print("Data types converted successfully!")

Data types converted successfully!


In [10]:

# INCONSISTENT VALUES CHECKING & CLEANING

# 1. Clean extra spaces in text columns
text_columns = ['Description', 'Country', 'StockCode']

for col in text_columns:
    df[col] = df[col].astype('string').str.strip()
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)

# 2. Standardize Country names
df['Country'] = df['Country'].str.title()

# 3. Check negative Quantity
negative_quantity = (df['Quantity'] < 0).sum()

# 4. Check zero and negative UnitPrice
zero_price = (df['UnitPrice'] == 0).sum()
negative_price = (df['UnitPrice'] < 0).sum()

# 5. Display unusual values before deciding what to do
print("========== INCONSISTENT VALUES REPORT ==========")

print("\nNegative Quantity:", negative_quantity)
print("Zero UnitPrice:", zero_price)
print("Negative UnitPrice:", negative_price)

print("\nCountry Names:")
print(df['Country'].unique())

# 6. Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# 7. Check duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# 8. Final dataset information
print("\nFinal Dataset Shape:", df.shape)

print("\nFinal Data Types:")
print(df.dtypes)

print("\nCleaning and checking completed!")

========== INCONSISTENT VALUES REPORT ==========

Negative Quantity: 10587
Zero UnitPrice: 2510
Negative UnitPrice: 2

Country Names:
<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'Eire',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'Usa', 